# EDA for Covariates

In [29]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)

# PUT GITHUB ROOT PATH HERE
root_dir = "/Users/albert/Library/CloudStorage/OneDrive-andrew.cmu.edu/Documents/CMU 26-27/Undergraduate Research/Victimization/Code/capstone_victimization"
dir = root_dir + "/Data and Resources"
os.chdir(dir)
data = pd.read_csv(os.path.join(dir, "final_data.csv"))
print(data.shape)
print(data.columns)

(8984, 26)
Index(['respondent_id', 'timely_college', 'violent_victimization_18_24',
       'earliest_arrest_before_18', 'total_number_of_incarcerations',
       'r_threatened_to_be_hurt_at_school', 'r_in_a_fight_at_school',
       'days_r_absent_from_school', 'teachers_good',
       'discipline_is_fair_agree_disagree', 'percent_of_peers_who_smoke',
       'percent_of_peers_who_get_drunk_1_times_a_month',
       'percent_of_peers_belong_to_a_gang',
       'percent_of_peers_who_have_had_sex', 'bullying_victim_before_age_12',
       'r_victim_of_bullying_12_18_years', 'sex',
       'r_ever_live_through_hard_times', 'household_net_worth',
       'household_size', 'biological_father_education',
       'biological_mother_education', 'race_ethnicity',
       'asvab_math_verbal_score_percent', 'mother_serve_prison_sentence',
       'father_serve_prison_sentence'],
      dtype='str')


In [30]:
covariates = [
    col for col in data.columns
    if col not in ["respondent_id", "timely_college", "violent_victimization_18_24"]
]

### Descriptive Stats

In [31]:
# Your feature lists
numeric_features = [
    "total_number_of_incarcerations",
    "days_r_absent_from_school",
    "r_threatened_to_be_hurt_at_school",
    "r_in_a_fight_at_school",
    "household_net_worth",
    "household_size",
    "asvab_math_verbal_score_percent"
]

categorical_features = [
    "teachers_good",
    "discipline_is_fair_agree_disagree",
    "percent_of_peers_who_smoke",
    "percent_of_peers_who_get_drunk_1_times_a_month",
    "percent_of_peers_belong_to_a_gang",
    "percent_of_peers_who_have_had_sex",
    "bullying_victim_before_age_12",
    "r_victim_of_bullying_12_18_years",
    "earliest_arrest_before_18",
    "sex",
    "r_ever_live_through_hard_times",
    "biological_father_education",
    "biological_mother_education",
    "race_ethnicity",
    "mother_serve_prison_sentence",
    "father_serve_prison_sentence"
]

In [32]:
covariate_data = data[numeric_features + categorical_features].copy()

In [33]:
# ---------------------------------------------------------
# 3. Clean NUMERIC variables
# ---------------------------------------------------------
# Convert to numeric and treat negative survey responses
# as missing.

for feature in numeric_features:
    
    covariate_data[feature] = pd.to_numeric(
        covariate_data[feature],
        errors="coerce"
    )
    
    covariate_data.loc[
        covariate_data[feature] < 0,
        feature
    ] = np.nan

In [34]:
# ---------------------------------------------------------
# 4. Clean CATEGORICAL variables
# ---------------------------------------------------------
# Some categorical variables are numeric codes while others
# (e.g. race_ethnicity) contain strings.
#
# We therefore:
#   - Try to identify numeric values
#   - Convert negative numeric values to NaN
#   - Leave string responses unchanged
#
# This avoids errors from comparing strings to integers.

for feature in categorical_features:
    
    # Identify which values can be interpreted as numbers
    numeric_values = pd.to_numeric(
        covariate_data[feature],
        errors="coerce"
    )
    
    # Only replace values that are actually numeric AND negative
    negative_values = numeric_values < 0
    
    covariate_data.loc[
        negative_values,
        feature
    ] = np.nan

In [35]:
# ---------------------------------------------------------
# 5. Numerical descriptive statistics
# ---------------------------------------------------------

numeric_stats = pd.DataFrame(index=numeric_features)

numeric_stats["N"] = (
    covariate_data[numeric_features]
    .notna()
    .sum()
)

numeric_stats["Missing"] = (
    covariate_data[numeric_features]
    .isna()
    .sum()
)

numeric_stats["Mean"] = (
    covariate_data[numeric_features]
    .mean()
)

numeric_stats["Std Dev"] = (
    covariate_data[numeric_features]
    .std()
)

numeric_stats["Min"] = (
    covariate_data[numeric_features]
    .min()
)

numeric_stats["Max"] = (
    covariate_data[numeric_features]
    .max()
)


# Display numerical statistics
print("NUMERICAL COVARIATES")
display(numeric_stats)

NUMERICAL COVARIATES


,N,Missing,Mean,Std Dev,Min,Max
total_number_of_incarcerations,8948,36,0.210326,0.758556,0.0,11.0
days_r_absent_from_school,8687,297,4.889375,7.588716,0.0,200.0
r_threatened_to_be_hurt_at_school,8869,115,0.977450,5.226506,0.0,99.0
r_in_a_fight_at_school,8872,112,0.401262,1.707949,0.0,50.0
household_net_worth,6193,2791,97492.775069,138216.143107,0.0,600000.0
household_size,8984,0,4.548976,1.540158,1.0,16.0
asvab_math_verbal_score_percent,7093,1891,45.317192,29.173908,0.0,100.0


In [42]:
# ---------------------------------------------------------
# Categorical counts: separate DataFrame for each variable
# ---------------------------------------------------------

categorical_dfs = {}

for feature in categorical_features:
    
    # Count each non-missing category
    counts = (
        covariate_data[feature]
        .value_counts(dropna=True)
        .sort_index()
        .reset_index()
    )
    
    # Rename columns
    counts.columns = ["Category", "Count"]
    
    # Add missing count
    missing = covariate_data[feature].isna().sum()
    
    counts.loc[len(counts)] = ["Missing", missing]
    
    # Store DataFrame
    categorical_dfs[feature] = counts

for feature in categorical_features:
    print(f"CATEGORICAL COUNTS FOR {feature}")
    display(categorical_dfs[feature])

CATEGORICAL COUNTS FOR teachers_good


,Category,Count
0,1.0,1573
1,2.0,6192
2,3.0,964
3,4.0,236
4,Missing,19


CATEGORICAL COUNTS FOR discipline_is_fair_agree_disagree


,Category,Count
0,1.0,1469
1,2.0,4932
2,3.0,1949
3,4.0,593
4,Missing,41


CATEGORICAL COUNTS FOR percent_of_peers_who_smoke


...

,Category,Count
0,Asian,156
1,Black,2334
2,Hispanic,1899
3,Other,182
4,White,4413
5,Missing,0


CATEGORICAL COUNTS FOR mother_serve_prison_sentence


,Category,Count
0,0.0,6976
1,1.0,94
2,Missing,1914


CATEGORICAL COUNTS FOR father_serve_prison_sentence


,Category,Count
0,0.0,6079
1,1.0,430
2,Missing,2475


In [37]:
# ---------------------------------------------------------
# 7. Categorical missing counts
# ---------------------------------------------------------

categorical_missing = pd.DataFrame({
    "Variable": categorical_features,
    "Missing": [
        covariate_data[feature].isna().sum()
        for feature in categorical_features
    ]
})


# Display missing counts
print("CATEGORICAL MISSING")
display(categorical_missing)

CATEGORICAL MISSING


,Variable,Missing
0,teachers_good,19
1,discipline_is_fair_agree_disagree,41
2,percent_of_peers_who_smoke,113
3,percent_of_peers_who_get_drunk_1_times_a_month,185
4,percent_of_peers_belong_to_a_gang,172
5,percent_of_peers_who_have_had_sex,5019
6,bullying_victim_before_age_12,151
7,r_victim_of_bullying_12_18_years,2025
8,earliest_arrest_before_18,0
9,sex,0


### Missingness

In [38]:
data[covariates].isna().sum()

earliest_arrest_before_18                            0
total_number_of_incarcerations                       0
r_threatened_to_be_hurt_at_school                    0
r_in_a_fight_at_school                               0
days_r_absent_from_school                            0
teachers_good                                        0
discipline_is_fair_agree_disagree                    0
percent_of_peers_who_smoke                           0
percent_of_peers_who_get_drunk_1_times_a_month       0
percent_of_peers_belong_to_a_gang                    0
percent_of_peers_who_have_had_sex                    0
bullying_victim_before_age_12                        0
r_victim_of_bullying_12_18_years                  2025
sex                                                  0
r_ever_live_through_hard_times                       0
household_net_worth                                  0
household_size                                       0
biological_father_education                          0
biological

### Check Asian counts (small?)

In [44]:
# Filter to Asian respondents
asian_data = data[
    data["race_ethnicity"] == "Asian"
].copy()

# Count the four combinations
asian_counts = pd.crosstab(
    asian_data["timely_college"],
    asian_data["violent_victimization_18_24"]
)

print("Counts for Asian respondents:")
asian_counts

Counts for Asian respondents:


violent_victimization_18_24,0.0,1.0
timely_college,,
0.0,42,1
1.0,95,4
